# Module 03: Lý thuyết bay / Theory of Flight
**AE2.008 · USTH: Aerodynamics and Flight Mechanics**

🇻🇳 Giải lại **Anderson 5.4, 5.5, 8.1, 8.7, 8.11, 11.4**, tính bảng hệ số tải và tốc độ thất tốc trong vòng bay, kiểm tốc độ âm thanh của EASA (661 kt và 574 kt) rồi `assert` các đáp án quiz.

🇬🇧 Re-solves **Anderson 5.4, 5.5, 8.1, 8.7, 8.11, 11.4**, builds the load-factor and turn-stall table, checks the EASA speeds of sound (661 kt and 574 kt) and asserts the quiz answers.

In [ ]:
import math
G0, R_AIR, GAMMA = 9.80665, 287.05287, 1.4
P0, T0, RHO0 = 101325.0, 288.15, 1.225
FT, PSI = 0.3048, 6894.757

def isa(h_m):
    """ISA 0-20 km: (T [K], p [Pa], rho [kg/m3]). Troposphere lapse -6.5 K/km, isothermal 216.65 K above 11 km."""
    if h_m <= 11000:
        L = -0.0065; T = T0 + L*h_m; p = P0*(T/T0)**(-G0/(L*R_AIR))
    else:
        T = 216.65; p11 = P0*(T/T0)**(-G0/(-0.0065*R_AIR)); p = p11*math.exp(-G0*(h_m-11000)/(R_AIR*T))
    return T, p, p/(R_AIR*T)

def isa_ft(h_ft): return isa(h_ft*FT)

def a_sound(T, R=R_AIR, g=GAMMA): return math.sqrt(g*R*T)
def pitot_ratio(M, g=GAMMA): return (1+(g-1)/2*M*M)**(g/(g-1))
def bisect(f, lo, hi, n=200):
    for _ in range(n):
        mid = (lo+hi)/2
        if f(lo)*f(mid) <= 0: hi = mid
        else: lo = mid
    return (lo+hi)/2


## 1. Piper Cherokee (Anderson 5.4, 5.5)

In [ ]:
S, b, W = 170.0, 32.0, 2450.0; AR = b*b/S
V = 120*5280/3600; q = 0.5*0.002377*V**2; CL = W/(q*S)
a0, tau, aL0 = 0.1033, 0.12, -3.0
a = a0/(1 + 57.3*a0/(math.pi*AR)*(1+tau)); alpha = CL/a + aL0
e = 0.64; CDi = CL**2/(math.pi*e*AR); Di = CDi*q*S
print(f"AR = {AR:.3f}, V = {V:.0f} ft/s, q = {q:.2f}, CL = {CL:.4f}, a = {a:.4f}/deg, alpha = {alpha:.2f} deg, CDi = {CDi:.5f}, Di = {Di:.1f} lb")
assert abs(CL-0.3915) < 5e-4 and abs(alpha-2.12) < 0.01 and abs(Di-79.2) < 0.2

## 2. Tốc độ âm thanh và Mach / Speed of sound and Mach number

In [ ]:
kt = 0.514444
a_sl = a_sound(T0)/kt; a_40 = a_sound(isa_ft(40000)[0])/kt
print(f"a(sea level) = {a_sl:.1f} kt (EASA: 661), a(40 000 ft) = {a_40:.1f} kt (EASA: 574)")
assert abs(a_sl-661) < 1 and abs(a_40-574) < 1
a81 = a_sound(230, 287); print("8.1 a =", round(a81, 1), "m/s"); assert abs(a81-304.0) < 0.1
M = bisect(lambda m: pitot_ratio(m)-1.555, 0.0, 1.0); V811 = M*a_sound(288, 287)
print(f"8.11 M = {M:.4f}, V = {V811:.1f} m/s"); assert abs(M-0.8199) < 1e-3 and abs(V811-278.9) < 0.2
V_bern = math.sqrt(2*0.555*101325/1.225); print(f"Bernoulli (wrong) V = {V_bern:.1f} m/s, error {abs(V_bern-V811)/V811*100:.1f}%")

## 3. Sóng xung kích thẳng / Normal shock (Anderson 8.7)

In [ ]:
g = GAMMA; M1 = 2.6
M2 = math.sqrt(((g-1)*M1**2+2)/(2*g*M1**2-(g-1))); p = 1+2*g/(g+1)*(M1**2-1)
rho = (g+1)*M1**2/((g-1)*M1**2+2); T = p/rho
print(f"M2 = {M2:.4f}, p2/p1 = {p:.3f}, rho2/rho1 = {rho:.3f}, T2/T1 = {T:.3f}, T2 = {T*288:.1f} K")
assert abs(M2-0.5039) < 1e-3 and abs(p-7.72) < 1e-3 and M2 < 1

## 4. Số Mach tới hạn / Critical Mach number (Anderson 11.4)

In [ ]:
def cp_crit(M, g=GAMMA): return 2/(g*M*M)*(((1+(g-1)/2*M*M)/((g+1)/2))**(g/(g-1))-1)
Mcr = bisect(lambda m: -0.41/math.sqrt(1-m*m)-cp_crit(m), 0.3, 0.99)
print("Prandtl-Glauert Mcr =", round(Mcr, 4)); assert abs(Mcr-0.7436) < 1e-3

## 5. Hệ số tải trong vòng bay / Load factor in a level turn

In [ ]:
print("bank  n     sqrt(n)  stall-speed increase")
for phi in (0, 30, 45, 60, 70):
    n = 1/math.cos(math.radians(phi)); print(f"{phi:>3}  {n:5.2f}  {math.sqrt(n):6.3f}  {(math.sqrt(n)-1)*100:5.0f}%")
assert abs(1/math.cos(math.radians(60))-2) < 1e-12
assert abs(60*math.sqrt(2)-84.85) < 0.01               # quiz: 60 kt stall -> ~85 kt at 60 deg bank
assert abs(3.8*1.5-5.7) < 1e-12                        # ultimate = 1.5 x limit (EASA p. 3.9)
assert abs(0.8*a_40-459) < 1                           # M 0.8 at 40 000 ft ~ 459 kt TAS
assert 36000/3000 == 12                                # glide ratio quiz

## 6. Kiểm quiz / Quiz self-check

In [ ]:
print("All Module 03 checks passed.")

## 7. Câu bổ sung / Extra questions

In [ ]:
assert abs((math.sqrt(1/math.cos(math.radians(45)))-1)*100-18.9) < 0.1
a20 = a_sound(isa_ft(20000)[0])/kt; print("a(20 000 ft) =", round(a20), "kt"); assert abs(a20-614) < 2
g = GAMMA; M1 = 2.0
M2 = math.sqrt(((g-1)*M1**2+2)/(2*g*M1**2-(g-1))); p = 1+2*g/(g+1)*(M1**2-1); assert abs(M2-0.5774) < 1e-3 and abs(p-4.5) < 1e-9
assert 2000*15 == 30000
assert abs(math.degrees(math.acos(1/2.5))-66.42) < 0.01
print("Extra Module 03 checks passed.")